# Working with FleXgeo2 results in memory

This notebook keeps the analysis in memory and shows how to ask common questions of the FleXgeo2 dataframes. It also demonstrates the lower-level `GeometryService` workflow that powers the high-level app.

In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "pdb2lj5.pdb").is_file():
            return candidate
    raise FileNotFoundError("Could not find the FleXgeo2 repo root containing pyproject.toml and pdb2lj5.pdb")


REPO_ROOT = find_repo_root()
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

try:
    import matplotlib.pyplot as plt
    import melodia_py  # noqa: F401
    import pandas as pd
    from flexgeo2 import AnalysisConfig, FlexGeo2App, OutputConfig
    from flexgeo2.geometry import GeometryService
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        f"Missing dependency: {exc.name}. Install the project in your notebook kernel with `pip install -e .` "
        "or, if you use uv, run `uv sync` and select the project environment as the Jupyter kernel."
    ) from exc

PDB_FILE = REPO_ROOT / "pdb2lj5.pdb"
print(f"Using {PDB_FILE}")

## Run without writing files

Use `OutputConfig(write_files=False)` when you want the dataframes but do not want FleXgeo2 to create CSV or PNG files.

In [ ]:
result = FlexGeo2App().run(
    AnalysisConfig(
        pdb_file=PDB_FILE,
        chains=["A"],
        n_jobs=1,
        output=OutputConfig(write_files=False),
    )
)

print(type(result.outputs).__name__)
result.residue_summary_df.head()

## The lower-level geometry flow

`GeometryService` is useful when you want to control each transformation step yourself.

In [ ]:
geometry = GeometryService()
raw_df = geometry.load_structure(PDB_FILE, n_jobs=1)
chain_a_df = geometry.filter_chains(raw_df, ["A"])
chain_a_df = geometry.normalize(chain_a_df)
residue_summary_df = geometry.summarize(chain_a_df)
model_summary_df, overall_model_summary_df = geometry.build_model_summary(chain_a_df, residue_summary_df)

print(f"Raw shape: {chain_a_df.shape}")
print(f"Residue summary shape: {residue_summary_df.shape}")
print(f"Model summary shape: {model_summary_df.shape}")

## Find residues with the most ensemble variation

Large standard deviations highlight residues whose geometry varies more across the submitted models.

In [ ]:
top_curvature_variability = residue_summary_df.sort_values("curvature_std", ascending=False).head(10)
top_curvature_variability[["chain", "residue_label", "curvature_mean", "curvature_std", "models"]]

In [ ]:
top_torsion_variability = residue_summary_df.sort_values("torsion_std", ascending=False).head(10)
top_torsion_variability[["chain", "residue_label", "torsion_mean", "torsion_std", "models"]]

## Find models that differ most from the ensemble mean

The mean absolute deviation columns summarize how far each model is from the ensemble-average residue profile.

In [ ]:
overall_model_summary_df.assign(
    combined_mean_abs_deviation=lambda df: df["curvature_mean_abs_deviation"] + df["torsion_mean_abs_deviation"]
).sort_values("combined_mean_abs_deviation", ascending=False).head(10)

## Plot one residue across the ensemble

Here we choose the residue with the largest torsion standard deviation and plot its curvature/torsion points for every model.

In [ ]:
selected = top_torsion_variability.iloc[0]
selected_residue = chain_a_df[chain_a_df["order"] == selected["order"]].copy()

ax = selected_residue.plot.scatter(
    x="curvature",
    y="torsion",
    figsize=(6, 5),
    alpha=0.75,
    title=f"Chain {selected['chain']} {selected['residue_label']} across models",
)
ax.set_xlabel("Curvature")
ax.set_ylabel("Torsion")
plt.show()